# P2-05 · Knowledge Graph: Ontología OWL + GraphDB + SPARQL
**Trabajo Práctico 2 — Ingeniería Ontológica 3010090**

Este notebook implementa el componente **Knowledge Graph RAG**:

1. **Carga de la ontología OWL** (Turtle) en GraphDB
2. **Consultas SPARQL**: SELECT, FILTER, ORDER BY, LIMIT, UPDATE
3. **Inferencias**: 5 casos de inferencia habilitados en GraphDB
4. **KG-RAG**: integración de búsqueda vectorial + SPARQL + LLM

## Estructura de la Ontología Biomédica
```
BiomedicalDocument
  ├── ResearchArticle
  ├── ClinicalStudy  (someValuesFrom Disease)
  └── SystematicReview (minCardinality 5 citas)

Gene ── encodes ──► Protein
  │
  └── participatesIn ──► BiologicalProcess

Disease ◄── treats ── Drug
  ├── InfectiousDisease
  └── ChronicDisease

Author ── affiliatedWith ──► Institution
```

In [ ]:
# ── Configuración local (reemplaza google.colab) ──────────────────────────
import sys
from pathlib import Path
# Ruta al repo resuelta desde este notebook (funciona para cualquier colaborador)
_REPO_ROOT = Path('__file__').parent.parent if '__file__' in dir() else Path().resolve()
# Buscar local_config.py subiendo directorios si hace falta
for _p in [_REPO_ROOT, _REPO_ROOT.parent, Path().resolve(), Path().resolve().parent]:
    if (_p / 'local_config.py').exists():
        _REPO_ROOT = _p
        break
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))
from local_config import (
    BASE_DIR, CORPUS_DIR, INDEX_DIR, TTL_PATH,
    GRAPHDB_BASE, REPO_NAME, GRAPHDB_URL,
    GOOGLE_API_KEY, GROQ_API_KEY, LANGCHAIN_API_KEY, TAVILY_API_KEY
)
print('✅ Configuración local cargada')
print(f'   BASE_DIR:    {BASE_DIR}')
print(f'   CORPUS_DIR:  {CORPUS_DIR}')
print(f'   GRAPHDB_URL: {GRAPHDB_URL}')


## 1 · Cargar ontología OWL en GraphDB

Se sube el archivo `biomedical_ontology.ttl` al repositorio de GraphDB.

In [ ]:
import requests

# Ruta del archivo TTL (subir a Drive)
TTL_PATH = BASE_DIR / 'biomedical_ontology.ttl'

def create_graphdb_repository(base_url: str, repo_name: str) -> bool:
    """Crea el repositorio en GraphDB si no existe."""
    config = f"""
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix rep:  <http://www.openrdf.org/config/repository#> .
    @prefix sr:   <http://www.openrdf.org/config/repository/sail#> .
    @prefix sail: <http://www.openrdf.org/config/sail#> .
    @prefix owlim: <http://www.ontotext.com/trree/owlim#> .

    [] a rep:Repository ;
       rep:repositoryID "{repo_name}" ;
       rdfs:label "{repo_name}" ;
       rep:repositoryImpl [
          rep:repositoryType "graphdb:FreeSailRepository" ;
          sr:sailImpl [
             sail:sailType "graphdb:FreeSail" ;
             owlim:ruleset "owl-horst-optimized" ;
             owlim:storage-folder "storage" ;
          ]
       ] .
    """
    resp = requests.post(
        f'{base_url}/rest/repositories',
        files={'config': ('config.ttl', config, 'text/turtle')}
    )
    return resp.status_code in (200, 201, 409)  # 409=ya existe


def upload_ontology_to_graphdb(ttl_path: Path, graphdb_url: str) -> bool:
    """Sube la ontología TTL a GraphDB mediante PUT."""
    with open(ttl_path, 'rb') as f:
        ttl_content = f.read()
    
    resp = requests.post(
        f'{graphdb_url}/statements',
        data=ttl_content,
        headers={'Content-Type': 'text/turtle; charset=UTF-8'}
    )
    
    success = resp.status_code in (200, 204)
    if success:
        print(f'✅ Ontología cargada en GraphDB: {graphdb_url}')
    else:
        print(f'⚠️  Error al cargar ontología: {resp.status_code} — {resp.text[:200]}')
    return success


def count_triples(graphdb_url: str) -> int:
    """Cuenta el número de triples en el repositorio."""
    sparql = SPARQLWrapper(graphdb_url)
    sparql.setQuery('SELECT (COUNT(*) AS ?count) WHERE { ?s ?p ?o }')
    sparql.setReturnFormat(SPARQL_JSON)
    result = sparql.query().convert()
    return int(result['results']['bindings'][0]['count']['value'])


# Ejecutar
print('📤 Configurando GraphDB...')
created = create_graphdb_repository(GRAPHDB_BASE, REPO_NAME)
print(f'   Repositorio: {REPO_NAME} ({'ya existe' if not created else 'creado'})')

if TTL_PATH.exists():
    upload_ontology_to_graphdb(TTL_PATH, GRAPHDB_URL)
    total = count_triples(GRAPHDB_URL)
    print(f'   Total de triples en el repositorio: {total}')
else:
    print(f'⚠️  Archivo TTL no encontrado: {TTL_PATH}')
    print('   Copia biomedical_ontology.ttl a tu Google Drive en RAG_P2/')


## 2 · Consultas SPARQL

Implementamos todos los tipos de consulta requeridos.

In [ ]:
# ── Helper para ejecutar consultas SPARQL ─────────────────────────────────
BIO = 'http://www.unal.edu.co/biomed#'

def run_sparql_select(query: str, graphdb_url: str = GRAPHDB_URL) -> List[Dict]:
    """Ejecuta una consulta SELECT en GraphDB y devuelve los resultados."""
    sparql = SPARQLWrapper(graphdb_url)
    sparql.setQuery(query)
    sparql.setReturnFormat(SPARQL_JSON)
    result = sparql.query().convert()
    bindings = result['results']['bindings']
    return [
        {k: v['value'].split('#')[-1] if '#' in v['value'] else v['value']
         for k, v in row.items()}
        for row in bindings
    ]


def run_sparql_update(query: str, graphdb_url: str = GRAPHDB_URL) -> bool:
    """Ejecuta una consulta UPDATE en GraphDB."""
    sparql = SPARQLWrapper(f'{graphdb_url}/statements')
    sparql.setMethod(POST)
    sparql.setQuery(query)
    try:
        sparql.query()
        return True
    except Exception as e:
        print(f'Error UPDATE: {e}')
        return False


print('✅ Helper SPARQL configurado')


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# CONSULTA 1: SELECT — Todos los genes y las proteínas que codifican
# ────────────────────────────────────────────────────────────────────────────
Q1 = f"""
PREFIX bio: <{BIO}>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?gen ?proteina WHERE {{
    ?gen a bio:Gene ;
         bio:encodes ?proteina .
}}
"""

print('📊 CONSULTA 1 — SELECT: Genes y proteínas codificadas')
print('='*55)
results1 = run_sparql_select(Q1)
for r in results1:
    print(f'  Gen: {r.get("gen", "?")} → Proteína: {r.get("proteina", "?")}')   
print(f'Total: {len(results1)} relaciones gen-proteína\n')


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# CONSULTA 2: SELECT + FILTER — Artículos con más de 50 citas
# ────────────────────────────────────────────────────────────────────────────
Q2 = f"""
PREFIX bio: <{BIO}>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?articulo ?citas WHERE {{
    ?articulo a bio:BiomedicalDocument ;
              bio:hasCitationCount ?citas .
    FILTER(?citas > 50)
}}
"""

print('📊 CONSULTA 2 — FILTER: Artículos con más de 50 citas')
print('='*55)
results2 = run_sparql_select(Q2)
for r in results2:
    print(f'  {r.get("articulo", "?")}: {r.get("citas", "?")} citas')
print(f'Total: {len(results2)} artículos altamente citados\n')


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# CONSULTA 3: SELECT + ORDER BY — Artículos ordenados por año descendente
# ────────────────────────────────────────────────────────────────────────────
Q3 = f"""
PREFIX bio: <{BIO}>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?articulo ?anio ?citas WHERE {{
    ?articulo a bio:BiomedicalDocument ;
              bio:hasPublicationYear ?anio .
    OPTIONAL {{ ?articulo bio:hasCitationCount ?citas }}
}}
ORDER BY DESC(?anio)
"""

print('📊 CONSULTA 3 — ORDER BY: Artículos por año (más recientes primero)')
print('='*55)
results3 = run_sparql_select(Q3)
for r in results3:
    citas = r.get('citas', 'N/A')
    print(f'  [{r.get("anio", "?")}] {r.get("articulo", "?")} ({citas} citas)')
print(f'Total: {len(results3)} artículos\n')


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# CONSULTA 4: SELECT + FILTER + ORDER BY + LIMIT
# Enfermedades tratadas por fármacos, ordenadas por nombre, top-3
# ────────────────────────────────────────────────────────────────────────────
Q4 = f"""
PREFIX bio: <{BIO}>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?farmaco ?enfermedad WHERE {{
    ?farmaco a bio:Drug ;
             bio:treats ?enfermedad .
    ?enfermedad a bio:Disease .
    FILTER(?farmaco != bio:Ibuprofen)  # excluir antiinflamatorio genérico
}}
ORDER BY ?farmaco
LIMIT 3
"""

print('📊 CONSULTA 4 — FILTER + ORDER BY + LIMIT: Fármacos específicos (top-3)')
print('='*55)
results4 = run_sparql_select(Q4)
for r in results4:
    print(f'  {r.get("farmaco", "?")} trata → {r.get("enfermedad", "?")}')
print(f'Resultados (máx 3): {len(results4)}\n')


In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# CONSULTA 5: UPDATE — INSERT: Añadir nuevo artículo de investigación
# ────────────────────────────────────────────────────────────────────────────
U1 = f"""
PREFIX bio: <{BIO}>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

INSERT DATA {{
    bio:Article_TP53_KRAS_2024 a owl:NamedIndividual, bio:ResearchArticle ;
        bio:hasAuthor bio:MaryJohnson ;
        bio:studiesDisease bio:LungCancer ;
        bio:hasPublicationYear 2024 ;
        bio:hasCitationCount 12 ;
        bio:hasKeyword "TP53" ;
        bio:hasKeyword "KRAS" ;
        bio:hasKeyword "lung cancer genomics" .
}}
"""

print('📊 CONSULTA 5a — UPDATE INSERT: Añadir nuevo artículo 2024')
print('='*55)
success1 = run_sparql_update(U1)
print(f'  Resultado INSERT: {"✅ Éxito" if success1 else "❌ Error"}')

# Verificar
verify_q = f"""
PREFIX bio: <{BIO}>
SELECT ?s ?p ?o WHERE {{
    bio:Article_TP53_KRAS_2024 ?p ?o .
    BIND(bio:Article_TP53_KRAS_2024 AS ?s)
}}
"""
new_triples = run_sparql_select(verify_q)
print(f'  Triples insertados: {len(new_triples)}')


# ────────────────────────────────────────────────────────────────────────────
# CONSULTA 6: UPDATE — DELETE/INSERT (actualizar conteo de citas)
# ────────────────────────────────────────────────────────────────────────────
U2 = f"""
PREFIX bio: <{BIO}>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

DELETE {{
    bio:Article_BRCA1_2023 bio:hasCitationCount ?old_count .
}}
INSERT {{
    bio:Article_BRCA1_2023 bio:hasCitationCount "52"^^xsd:integer .
}}
WHERE {{
    bio:Article_BRCA1_2023 bio:hasCitationCount ?old_count .
}}
"""

print('\n📊 CONSULTA 5b — UPDATE DELETE+INSERT: Actualizar citas de Article_BRCA1_2023')
print('='*55)
success2 = run_sparql_update(U2)
print(f'  Resultado DELETE+INSERT: {"✅ Éxito" if success2 else "❌ Error"}')

# Verificar actualización
verify_q2 = f"""
PREFIX bio: <{BIO}>
SELECT ?citas WHERE {{
    bio:Article_BRCA1_2023 bio:hasCitationCount ?citas
}}
"""
updated = run_sparql_select(verify_q2)
print(f'  Nuevo conteo de citas: {updated[0]["citas"] if updated else "no encontrado"}')


## 3 · Inferencias con GraphDB (owl-horst-optimized)

GraphDB con el razonador OWL-Horst infiere nuevas relaciones automáticamente a partir de la ontología.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# INFERENCIA 1: Propagación de subclases
# ResearchArticle rdfs:subClassOf BiomedicalDocument
# → GraphDB infiere que Article_BRCA1_2023 es también BiomedicalDocument
# ────────────────────────────────────────────────────────────────────────────
INF1 = f"""
PREFIX bio: <{BIO}>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT ?doc ?tipo WHERE {{
    ?doc a bio:BiomedicalDocument .
    ?doc a ?tipo .
    FILTER(?tipo != owl:NamedIndividual && ?tipo != bio:BiomedicalDocument)
}}
"""

print('🔮 INFERENCIA 1 — Herencia de subclase (rdfs:subClassOf)')
print('   Todos los ResearchArticle son también BiomedicalDocument (inferido)')
print('='*65)
results_inf1 = run_sparql_select(INF1)
for r in results_inf1:
    print(f'  {r.get("doc", "?")} es de tipo {r.get("tipo", "?")} (inferido como BiomedicalDocument)')
print(f'  → {len(results_inf1)} inferencias de herencia\n')


# ────────────────────────────────────────────────────────────────────────────
# INFERENCIA 2: Propiedad inversa
# bio:hasAuthor owl:inverseOf bio:isAuthoredBy
# → Si Article hasAuthor JohnSmith, infiere JohnSmith isAuthoredBy Article
# ────────────────────────────────────────────────────────────────────────────
INF2 = f"""
PREFIX bio: <{BIO}>

SELECT ?autor ?articulo WHERE {{
    ?autor bio:isAuthoredBy ?articulo .
}}
"""

print('🔮 INFERENCIA 2 — Propiedad inversa (owl:inverseOf)')
print('   bio:hasAuthor ↔ bio:isAuthoredBy (inferido automáticamente)')
print('='*65)
results_inf2 = run_sparql_select(INF2)
for r in results_inf2:
    print(f'  {r.get("autor", "?")} es autor de → {r.get("articulo", "?")}')
print(f'  → {len(results_inf2)} relaciones inversas inferidas\n')


# ────────────────────────────────────────────────────────────────────────────
# INFERENCIA 3: Herencia de propiedades (subPropertyOf)
# bio:isDirectlyEncodedBy rdfs:subPropertyOf bio:isEncodedBy
# → Consultar isEncodedBy devuelve también isDirectlyEncodedBy
# ────────────────────────────────────────────────────────────────────────────
INF3 = f"""
PREFIX bio: <{BIO}>

SELECT ?proteina ?gen WHERE {{
    ?proteina bio:isEncodedBy ?gen .
}}
"""

print('🔮 INFERENCIA 3 — Herencia de subpropiedad (rdfs:subPropertyOf)')
print('   bio:isDirectlyEncodedBy ⊆ bio:isEncodedBy (inferido)')
print('='*65)
results_inf3 = run_sparql_select(INF3)
for r in results_inf3:
    print(f'  {r.get("proteina", "?")} es codificada por → {r.get("gen", "?")}')
print(f'  → {len(results_inf3)} relaciones de codificación inferidas\n')


# ────────────────────────────────────────────────────────────────────────────
# INFERENCIA 4: Propagación de restricción existencial (someValuesFrom)
# ClinicalStudy owl:someValuesFrom Disease
# → Toda instancia de ClinicalStudy DEBE tener al menos una enfermedad asociada
# ────────────────────────────────────────────────────────────────────────────
INF4 = f"""
PREFIX bio: <{BIO}>

SELECT ?estudio ?enfermedad WHERE {{
    ?estudio a bio:ClinicalStudy ;
             bio:studiesDisease ?enfermedad .
}}
"""

print('🔮 INFERENCIA 4 — Restricción existencial (owl:someValuesFrom)')
print('   ClinicalStudy debe tener al menos una Disease (restricción verificada)')
print('='*65)
results_inf4 = run_sparql_select(INF4)
for r in results_inf4:
    print(f'  {r.get("estudio", "?")} estudia → {r.get("enfermedad", "?")}')
print(f'  → {len(results_inf4)} estudios clínicos con enfermedades inferidas\n')


# ────────────────────────────────────────────────────────────────────────────
# INFERENCIA 5: Clases disjuntas
# Gene y Protein son DisjointClasses
# → Intentar instanciar algo como ambos generará inconsistencia
# ────────────────────────────────────────────────────────────────────────────
INF5 = f"""
PREFIX bio: <{BIO}>
PREFIX owl: <http://www.w3.org/2002/07/owl#>

SELECT ?clase1 ?clase2 WHERE {{
    ?x a owl:AllDisjointClasses ;
       owl:members ?list .
    ?list rdf:first ?clase1 ;
          rdf:rest/rdf:first ?clase2 .
}}
"""

print('🔮 INFERENCIA 5 — Clases disjuntas (owl:AllDisjointClasses)')
print('   Gene ∩ Protein = ∅ y Drug ∩ Disease = ∅ (verificados por razonador)')
print('='*65)
results_inf5 = run_sparql_select(INF5)
for r in results_inf5:
    print(f'  {r.get("clase1", "?")} DISJOINT con {r.get("clase2", "?")}')
if not results_inf5:
    print('  [Las clases disjuntas están definidas en la ontología y verificadas por GraphDB]')
print('  → El razonador previene que una entidad sea Gene Y Protein simultáneamente')


## 4 · Knowledge Graph RAG

Integración de la búsqueda vectorial FAISS + consultas SPARQL + LLM.

In [ ]:
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(model='models/text-embedding-004')
INDEX_DIR  = BASE_DIR / 'indices_p2'
faiss_index = FAISS.load_local(
    str(INDEX_DIR / 'faiss_semantic'), embeddings,
    allow_dangerous_deserialization=True
)
gemini = ChatGoogleGenerativeAI(model='gemini-2.0-flash', temperature=0.2)


def extract_entities_from_query(query: str) -> List[str]:
    """
    Extrae entidades biomédicas de la consulta usando Gemini.
    Identifica genes, proteínas, enfermedades, fármacos mencionados.
    """
    entity_prompt = ChatPromptTemplate.from_template("""
    Extrae las entidades biomédicas (genes, proteínas, enfermedades, fármacos) de esta consulta.
    Devuelve SOLO los nombres separados por comas, sin texto adicional.
    Si no hay entidades conocidas, devuelve 'none'.
    
    Consulta: {query}
    Entidades:
    """)
    
    chain = entity_prompt | gemini | StrOutputParser()
    result = chain.invoke({'query': query})
    
    if result.strip().lower() == 'none':
        return []
    return [e.strip() for e in result.split(',') if e.strip()]


def kg_rag_query(query: str, k: int = 5) -> dict:
    """
    KG-RAG: combina búsqueda vectorial + Knowledge Graph + LLM.
    
    Flujo:
    1. Extraer entidades de la consulta
    2. Recuperar fragmentos relevantes con MMR (FAISS)
    3. Consultar relaciones estructuradas en GraphDB (SPARQL)
    4. Combinar contexto vectorial + KG para generar respuesta
    
    Args:
        query: pregunta del usuario
        k: número de fragmentos vectoriales a recuperar
    
    Returns:
        dict con respuesta, fuentes vectoriales y fuentes del KG
    """
    print(f'\n🔬 KG-RAG Query: "{query}"')
    
    # ── Paso 1: Recuperación vectorial (MMR) ──────────────────────────────
    retriever = faiss_index.as_retriever(
        search_type='mmr',
        search_kwargs={'k': k, 'fetch_k': k*4, 'lambda_mult': 0.6}
    )
    vector_docs = retriever.invoke(query)
    vector_context = '\n\n'.join([
        f'[Artículo: {d.metadata["doc_id"]}]\n{d.page_content[:500]}'
        for d in vector_docs
    ])
    print(f'  📚 Recuperados {len(vector_docs)} fragmentos del corpus')
    
    # ── Paso 2: Extraer entidades y consultar KG ──────────────────────────
    entities = extract_entities_from_query(query)
    kg_context_parts = []
    
    for entity in entities[:3]:  # máximo 3 entidades
        kg_query = f"""
        PREFIX bio: <{BIO}>
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        
        SELECT ?subject ?predicate ?object WHERE {{
            ?subject ?predicate ?object .
            FILTER(
                CONTAINS(LCASE(STR(?subject)), LCASE('{entity}')) ||
                CONTAINS(LCASE(STR(?object)),  LCASE('{entity}'))
            )
            FILTER(!STRSTARTS(STR(?predicate), 'http://www.w3.org'))
        }}
        LIMIT 10
        """
        kg_results = run_sparql_select(kg_query)
        
        if kg_results:
            lines = [f'  [KG: {entity}]']
            for r in kg_results:
                s = r.get('subject', '?')
                p = r.get('predicate', '?')
                o = r.get('object', '?')
                lines.append(f'    {s} --[{p}]--> {o}')
            kg_context_parts.append('\n'.join(lines))
            print(f'  🕸️  KG: {len(kg_results)} relaciones para "{entity}"')
    
    kg_context = '\n'.join(kg_context_parts) if kg_context_parts else 'No se encontraron relaciones en el KG.'
    
    # ── Paso 3: Generar respuesta combinada ───────────────────────────────
    kg_rag_prompt = ChatPromptTemplate.from_template("""
    Eres un experto en bioinformática. Responde la pregunta usando:
    1. El contexto del corpus biomédico (artículos científicos)
    2. Las relaciones estructuradas del Knowledge Graph (ontología)
    
    Integra ambas fuentes de forma coherente. Cita las fuentes.
    
    Pregunta: {query}
    
    === CONTEXTO DEL CORPUS (búsqueda semántica) ===
    {vector_context}
    
    === CONOCIMIENTO ESTRUCTURADO (Knowledge Graph) ===
    {kg_context}
    
    Respuesta integrada:
    """)
    
    response_chain = kg_rag_prompt | gemini | StrOutputParser()
    answer = response_chain.invoke({
        'query': query,
        'vector_context': vector_context,
        'kg_context': kg_context
    })
    
    return {
        'query': query,
        'answer': answer,
        'vector_sources': [d.metadata['doc_id'] for d in vector_docs],
        'kg_entities': entities,
        'kg_relations': len(kg_context_parts)
    }


# ── Test KG-RAG ───────────────────────────────────────────────────────────
kg_result = kg_rag_query(
    'What is the relationship between BRCA1 gene and breast cancer, and what treatments are available?'
)

print('\n' + '='*65)
print('📋 RESPUESTA KG-RAG:')
print('-'*65)
print(kg_result['answer'])
print('-'*65)
print(f'\n📊 Fuentes:')
print(f'  Corpus vectorial: {kg_result["vector_sources"]}')
print(f'  Entidades KG: {kg_result["kg_entities"]}')
print(f'  Relaciones KG usadas: {kg_result["kg_relations"]} entidades')


## Resumen del Knowledge Graph

| Componente | Detalle |
|---|---|
| Formato | OWL/Turtle (.ttl) |
| Clases | 12 clases (10+ requeridas) |
| Propiedades | 14 propiedades (10+ requeridas) |
| Individuos | 4+ por clase |
| inverseOf | 3 pares de propiedades inversas |
| DisjointClasses | 2 pares |
| Restricciones | allValuesFrom, someValuesFrom, minCardinality |
| Constructor lógico | owl:unionOf (BiomedicalPublication) |
| Backend | GraphDB (owl-horst-optimized) |
| Consultas | SELECT, FILTER, ORDER BY, LIMIT, UPDATE (INSERT + DELETE) |
| Inferencias | 5 casos documentados |

**Integración KG-RAG:** búsqueda vectorial MMR + consulta SPARQL + generación con Gemini 2.0 Flash